# AI Model Adversial Attack and Defense System

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import tensorflow as tf
from scipy.optimize import minimize
from collections import defaultdict

class AdversarialMLSecurityFramework:
    def __init__(self):
        self.target_model = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42)
        self.defense_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42)
        self.scaler = StandardScaler()
        self.attack_history = defaultdict(list)
        self.epsilon_bounds = {'fgsm': 0.3, 'pgd': 0.1, 'carlini': 0.5}

    def generate_fgsm_attack(self, X, y, epsilon=0.3):
        X_scaled = self.scaler.fit_transform(X)
        self.target_model.fit(X_scaled, y)

        adversarial_samples = []
        for i, sample in enumerate(X_scaled):
            gradient = self.approximate_gradient(sample, y[i])
            perturbation = epsilon * np.sign(gradient)
            adversarial_sample = sample + perturbation
            adversarial_samples.append(np.clip(adversarial_sample, -3, 3))

        return np.array(adversarial_samples)

    def approximate_gradient(self, sample, true_label):
        delta = 1e-4
        gradient = np.zeros_like(sample)
        base_prediction = self.target_model.predict_proba([sample])[0]

        for i in range(len(sample)):
            sample_plus = sample.copy()
            sample_plus[i] += delta
            pred_plus = self.target_model.predict_proba([sample_plus])[0]
            gradient[i] = (pred_plus[true_label] - base_prediction[true_label]) / delta

        return gradient

    def generate_pgd_attack(self, X, y, epsilon=0.1, alpha=0.01, iterations=10):
        X_scaled = self.scaler.transform(X)
        adversarial_samples = X_scaled.copy()

        for _ in range(iterations):
            for i, sample in enumerate(adversarial_samples):
                gradient = self.approximate_gradient(sample, y[i])
                sample += alpha * np.sign(gradient)
                perturbation = sample - X_scaled[i]
                perturbation = np.clip(perturbation, -epsilon, epsilon)
                adversarial_samples[i] = X_scaled[i] + perturbation

        return adversarial_samples

    def generate_carlini_wagner_attack(self, X, y, c=1.0, kappa=0, max_iter=100):
        X_scaled = self.scaler.transform(X)
        adversarial_samples = []

        for i, (sample, label) in enumerate(zip(X_scaled, y)):
            def objective(delta):
                perturbed = sample + delta
                prediction = self.target_model.predict_proba([perturbed])[0]
                target_class_prob = prediction[label]
                max_other_prob = max(prediction[j] for j in range(len(prediction)) if j != label)

                loss = max(0, max_other_prob - target_class_prob + kappa)
                regularization = c * np.sum(delta**2)
                return loss + regularization

            result = minimize(objective, np.zeros_like(sample), method='BFGS',
                            options={'maxiter': max_iter, 'disp': False})
            adversarial_samples.append(sample + result.x)

        return np.array(adversarial_samples)

    def implement_adversarial_training(self, X_train, y_train, attack_ratio=0.3):
        clean_samples = int(len(X_train) * (1 - attack_ratio))
        X_clean = X_train[:clean_samples]
        y_clean = y_train[:clean_samples]

        X_attack = X_train[clean_samples:]
        y_attack = y_train[clean_samples:]

        adversarial_samples = self.generate_fgsm_attack(X_attack, y_attack)
        X_combined = np.vstack([X_clean, adversarial_samples])
        y_combined = np.hstack([y_clean, y_attack])

        self.target_model.fit(self.scaler.fit_transform(X_combined), y_combined)
        return self.target_model

    def detect_adversarial_examples(self, X_test):
        X_scaled = self.scaler.transform(X_test)
        detection_features = []

        for sample in X_scaled:
            predictions = self.target_model.predict_proba([sample])[0]
            confidence = np.max(predictions)
            entropy = -np.sum(predictions * np.log(predictions + 1e-10))

            noise_level = np.std(sample)
            gradient_norm = np.linalg.norm(self.approximate_gradient(sample, np.argmax(predictions)))

            detection_features.append([confidence, entropy, noise_level, gradient_norm])

        detection_features = np.array(detection_features)
        self.defense_model.fit(detection_features, np.ones(len(detection_features)))
        anomaly_scores = self.defense_model.decision_function(detection_features)

        return anomaly_scores < np.percentile(anomaly_scores, 10)

    def evaluate_robustness(self, X_test, y_test):
        attacks = {
            'fgsm': self.generate_fgsm_attack(X_test, y_test),
            'pgd': self.generate_pgd_attack(X_test, y_test),
            'carlini': self.generate_carlini_wagner_attack(X_test, y_test)
        }

        results = {}
        for attack_name, adversarial_X in attacks.items():
            predictions = self.target_model.predict(adversarial_X)
            accuracy = accuracy_score(y_test, predictions)
            detected = self.detect_adversarial_examples(adversarial_X)
            detection_rate = np.mean(detected)

            results[attack_name] = {'accuracy': accuracy, 'detection_rate': detection_rate}

        return results